# Generating decays

This page shows the basic workflow: create a {py:class}`.Generator`, generate decay trees, and turn them into four-momentum arrays.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib_inline.backend_inline import set_matplotlib_formats

set_matplotlib_formats("svg")

## Creating a generator

The default {py:class}`.Generator` uses the decay table ({py:data}`.DECAY_FILE`) and particle table ({py:data}`.PARTICLE_TABLE`) that ship with the package. A seed makes the generated events reproducible. Creating a {py:class}`.Generator` reads the decay and particle tables and initializes Pythia8, which prints a lot of log output. That output is hidden on this page.

In [ ]:
from pyevtgen import Generator

generator = Generator(seed=42)

EvtGen keeps its tables and random engine in global state. A process can therefore hold only one {py:class}`.Generator`, and creating a second one raises an error:

In [ ]:
try:
    Generator()
except RuntimeError as error:
    print(error)

The package records which versions of EvtGen, HepMC3 and Pythia8 it was built with, and where its data files are:

In [ ]:
import pyevtgen

print("EvtGen ", pyevtgen.EVTGEN_VERSION)
print("HepMC3 ", pyevtgen.HEPMC3_VERSION)
print("Pythia8", pyevtgen.PYTHIA8_VERSION)
print(pyevtgen.DECAY_FILE.name, pyevtgen.PARTICLE_TABLE.name)

## Decay trees

{py:meth}`.Generator.decay` decays a particle, given by its EvtGen name or PDG ID, and returns the root {py:class}`.Particle` of the decay tree. Each {py:class}`.Particle` holds its name, PDG ID, four-momentum $(E, p_x, p_y, p_z)$ in GeV, and its {py:attr}`~.Particle.daughters`.

In [ ]:
def print_tree(particle, indent: int = 0) -> None:
    energy = particle.p4[0]
    print(f"{' ' * indent}{particle.name:<10} E = {energy:.3f} GeV")
    for daughter in particle.daughters:
        print_tree(daughter, indent + 2)

In [ ]:
event = generator.decay("D*+")
print_tree(event)

The stable particles at the leaves of the tree are returned by {py:meth}`.Particle.final_state`. Their four-momenta add up to that of the parent:

In [ ]:
final_state = event.final_state()
[particle.name for particle in final_state]

In [ ]:
np.sum([particle.p4 for particle in final_state], axis=0)

## Event samples

{py:meth}`.Generator.generate` produces many decays at once. When all events have the same final state, {py:func}`.final_state_momenta` stacks them into a NumPy array of shape `(n_events, n_final_state, 4)`. Here, the $D^0$ decays according to the full decay table, so we first select one decay channel:

In [ ]:
events = generator.generate("D0", n=20_000)
channel = ["K-", "pi+"]
selected = [e for e in events if [p.name for p in e.final_state()] == channel]
len(selected)

In [ ]:
from pyevtgen import final_state_momenta

momenta = final_state_momenta(selected)
momenta.shape

In this two-body decay at rest, both daughters have a fixed momentum, while their directions are distributed uniformly over all directions:

In [ ]:
np.linalg.norm(momenta[:, :, 1:], axis=2).std(axis=0)

In [ ]:
p_kaon = momenta[:, 0, 1:]
cos_theta = p_kaon[:, 2] / np.linalg.norm(p_kaon, axis=1)
phi = np.arctan2(p_kaon[:, 1], p_kaon[:, 0])

fig, axes = plt.subplots(ncols=2, figsize=(8, 3))
axes[0].hist(cos_theta, bins=50)
axes[0].set_xlabel(r"$\cos\theta_{K^-}$")
axes[1].hist(phi, bins=50)
axes[1].set_xlabel(r"$\phi_{K^-}$")
fig.tight_layout()
plt.show()

## Moving parents

By default, the parent decays at rest. Pass its four-momentum to generate decays in flight, for instance a $D^0$ with an energy of 10 GeV along the $z$ axis:

In [ ]:
mass = generator.properties("D0").mass
energy = 10.0
p4 = [energy, 0.0, 0.0, np.sqrt(energy**2 - mass**2)]
event = generator.decay("D0", p4=p4)
np.sum([particle.p4 for particle in event.final_state()], axis=0)

## Reproducibility

The {py:attr}`~.Generator.seed` property reseeds the random engine, so the same seed gives the same events:

In [ ]:
generator.seed = 7
first = generator.decay("D0").final_state()[0].p4
generator.seed = 7
second = generator.decay("D0").final_state()[0].p4
first == second